# N-gram + 拉普拉斯平滑 中文语言模型

基于搜狗新闻语料（`CN_Corpus/SogouC.reduced`，GBK 编码）实现的统计语言模型。

三个核心要素：
- **样本**：语料里的句子
- **词汇表大小**：v
- **拉普拉斯平滑系数**：k（防止没见过的搭配概率直接为 0）

核心公式：

$$p(w_i \mid w_{i-n+1}\cdots w_{i-1}) = \dfrac{\text{count}(w_{i-n+1}\cdots w_i) + k}{\text{count}(w_{i-n+1}\cdots w_{i-1}) + k \times v}$$

## 环境配置

```bash
cd homework-2026-11-21-lm
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
# 把 .venv 注册成 jupyter 内核
python -m ipykernel install --user --name ngram-venv --display-name "Python (.venv ngram)"
jupyter notebook ngram_lm.ipynb
```

运行时右上角选择内核 **Python (.venv ngram)**。

In [11]:
import os
import re
import math
import random
from collections import Counter
from pathlib import Path

import jieba
from tqdm.auto import tqdm

# 分词器小测试
list(jieba.cut("我爱自然语言处理"))

['我', '爱', '自然语言', '处理']

## 1. 加载语料

语料结构：`CN_Corpus/SogouC.reduced/Reduced/C0000xx/数字.txt`，**GBK 编码**。

- 用 `gb18030` 读取（gb2312/gbk 的超集），坏字节容错跳过
- 去掉网页残留标签，按 `。！？；` 等标点切分成句子
- 语料全量约 9000+ 篇，用 `MAX_FILES` 控制采样规模，方便快速跑通；调大可训练更充分的模型

In [12]:
CORPUS_DIR = Path("CN_Corpus/SogouC.reduced/Reduced")
MAX_FILES = 300        # 最多读多少篇文档；None 表示全量（较慢）
MIN_SENT_LEN = 2       # 过滤过短的句子

SENT_SPLIT_RE = re.compile(r"[。！？!?；;\n\r]+")
TAG_RE = re.compile(r"<[^>]+>")


def read_text(path: Path) -> str:
    """读取单个语料文件，gb18030 编码，容错跳过坏字节。"""
    return path.read_bytes().decode("gb18030", errors="ignore")


def split_sentences(text: str) -> list[str]:
    """去标签 -> 按标点切句 -> 去空白/去短句。"""
    text = TAG_RE.sub("", text)
    sents = [s.strip() for s in SENT_SPLIT_RE.split(text)]
    return [s for s in sents if len(s) >= MIN_SENT_LEN]


files = sorted(CORPUS_DIR.glob("*/*.txt"))
if MAX_FILES:
    files = files[:MAX_FILES]
print(f"共 {len(files)} 篇文档")

sentences = []
for f in tqdm(files, desc="读取语料"):
    sentences.extend(split_sentences(read_text(f)))

print(f"共 {len(sentences)} 个句子")
for s in sentences[:5]:
    print("-", s[:60])

共 300 篇文档


读取语料:   0%|          | 0/300 [00:00<?, ?it/s]

共 7709 个句子
- 本报记者陈雪频实习记者唐翔发自上海
- 一家刚刚成立两年的网络支付公司，它的目标是成为市值100亿美元的上市公司
- 这家公司叫做快钱，说这句话的是快钱的CEO关国光
- 他之前曾任网易的高级副总裁，负责过网易的上市工作
- 对于为什么选择第三方支付作为创业方向，他曾经对媒体这样说：“我能看到这个胡同对面是什么，别人只能看到这个胡同


## 2. 切词 + 加 `<s>` / `</s>` 标记

- 用 jieba 分词
- 句首加 `<s>`、句尾加 `</s>`，让模型知道句子从哪开始、到哪结束

In [13]:
MAX_SENT_LEN = 50  # 丢弃过长的句子，控制训练规模


def tokenize_sentence(sent: str) -> list[str]:
    """分词，句首加 1 个 <s>，句尾加 </s>。更高阶模型训练时会自动补足 <s>。"""
    words = [w for w in jieba.cut(sent) if w.strip()]
    return ["<s>"] + words + ["</s>"]


tokenized = []
for s in tqdm(sentences, desc="jieba 分词"):
    toks = tokenize_sentence(s)
    if len(toks) <= MAX_SENT_LEN:
        tokenized.append(toks)

print(f"保留 {len(tokenized)} 个句子")
for toks in tokenized[:3]:
    print("-", " ".join(toks))

jieba 分词:   0%|          | 0/7709 [00:00<?, ?it/s]

保留 6886 个句子
- <s> 本报记者 陈雪频 实习 记者 唐翔 发自 上海 </s>
- <s> 一家 刚刚 成立 两年 的 网络 支付 公司 ， 它 的 目标 是 成为 市值 100 亿美元 的 上市公司 </s>
- <s> 这家 公司 叫做 快 钱 ， 说 这句 话 的 是 快 钱 的 CEO 关国光 </s>


## 3 ~ 4. 数次数 → 算概率（训练 = 数数！）

实现一个通用的 `NGramLM`：

- 词表 `vocab`：训练语料里出现过的所有词（+ 兜底的 `<unk>`），大小即 v
- `ngram_counts`：n-gram 出现次数；`context_counts`：前缀（上下文）出现总次数
- 拉普拉斯平滑（k=1）：

$$p(w \mid ctx) = \dfrac{\text{count}(ctx, w) + k}{\text{count}(ctx) + k \times v}$$

In [14]:
class NGramLM:
    """带拉普拉斯平滑的 n-gram 语言模型（n >= 2）。"""

    def __init__(self, n: int = 2, k: float = 1.0):
        assert n >= 2, "n 需 >= 2（至少要有一个上文词做条件）"
        self.n = n
        self.k = k
        self.vocab: set = set()
        self.ngram_counts = Counter()   # {(ctx..., w): 次数}  —— 公式的「分子」
        self.context_counts = Counter() # {(ctx...): 前缀出现总次数}  —— 公式的「分母」

    # ---------- 训练：本质是数数 ----------
    #
    # 对应公式 p(w_i | w_{i-n+1}..w_{i-1}) = count(前缀+w) / count(前缀)
    # 训练不做任何「学习」，只是把语料里每个 n-gram 和每个前缀各数一遍：
    #   - ngram_counts[前缀 + 当前词] += 1   -> 分子 count(ctx, w)
    #   - context_counts[前缀] += 1          -> 分母 count(ctx)
    # 两者对同一个位置 i 各加一次，天然满足：
    #   context_counts[ctx] == sum(ngram_counts[ctx + w] for w in 词表)
    def fit(self, tokenized_sents: list) -> "NGramLM":
        n = self.n
        for toks in tokenized_sents:
            # --- 第 1 步：统一句首标记 ---
            # tokenize_sentence 只加了 1 个 <s>，但 n-gram 模型需要 n-1 个：
            # n=3 时第一个真实词的前缀要两个词（<s> <s>），不够就得补。
            # 先把输入里已有的 <s> 全部去掉（不管加了几个），再按本模型阶数
            # 补足 n-1 个 —— 同一份分词结果可以喂给任何阶数的模型。
            body = list(toks)
            while body and body[0] == "<s>":
                body.pop(0)
            seq = ["<s>"] * (n - 1) + body   # 例：n=3, body=[我,爱,NLP,</s>] -> [<s>,<s>,我,爱,NLP,</s>]

            # --- 第 2 步：统计词表 ---
            # vocab 的大小就是公式里的 v（平滑分母里的 k × v）
            self.vocab.update(seq)

            # --- 第 3 步：滑动窗口逐个数数 ---
            # i 从 n-1 开始：前 n-1 个 <s> 是「垫脚的上下文」，不作为预测目标；
            # 每个位置 i 上：
            #   ctx = seq[i-n+1 : i]  长度恰为 n-1 的前缀（即公式里的 w_{i-n+1}..w_{i-1}）
            #   seq[i]               当前要预测的词（即公式里的 w_i）
            for i in range(n - 1, len(seq)):
                ctx = tuple(seq[i - n + 1: i])          # 前缀窗口，长度 n-1
                self.ngram_counts[ctx + (seq[i],)] += 1 # 数完整 n 元组：count(ctx, w)
                self.context_counts[ctx] += 1           # 数前缀本身：count(ctx)
        # --- 第 4 步：兜底词 ---
        # 推理时遇到词表外的词统一映射成 <unk>，这样 prob() 查得到、
        # 概率不为 0；平滑再给没见过的搭配兜一层底。
        # 注意它不参与上面的数数（训练语料中没有生词），
        # 但要占 v 的一个名额，推理时概率分布才能归一化。
        self.vocab.add("<unk>")
        return self

    # ---------- 概率：次数 + 平滑 -> 条件概率 ----------
    def _norm(self, w: str) -> str:
        return w if w in self.vocab else "<unk>"

    def prob(self, w: str, *ctx: str) -> float:
        """p(w | ctx)，拉普拉斯平滑。"""
        assert len(ctx) == self.n - 1
        w = self._norm(w)
        ctx = tuple(self._norm(c) for c in ctx)
        num = self.ngram_counts[ctx + (w,)] + self.k
        den = self.context_counts[ctx] + self.k * len(self.vocab)
        return num / den

    def log_prob(self, w: str, *ctx: str) -> float:
        return math.log2(self.prob(w, *ctx))

    # ---------- 查候选：前文定了，下一个词有哪些选择 ----------
    def candidates(self, *ctx: str, top_k: int = 10):
        """返回 [(词, 概率), ...]，按概率从大到小。
        语料中没接过的词，平滑后概率都相同，合并展示为一项 <unk>(N 词)，
        固定排在末尾（它是上万个词的概率总和，排前面会挤掉真实候选）。"""
        assert len(ctx) == self.n - 1
        ctx = tuple(self._norm(c) for c in ctx)
        seen = {g[-1] for g in self.ngram_counts if g[:-1] == ctx}
        items = [(w, self.prob(w, *ctx)) for w in seen]
        items.sort(key=lambda x: -x[1])          # 见过的词之间按概率排序
        n_unseen = len(self.vocab) - len(seen)
        if n_unseen > 0:
            merged = (f"<unk>({n_unseen} 词)", self.prob("<unk>", *ctx) * n_unseen)
            items.append(merged)                 # 合并项固定垫底
            if top_k is not None and len(items) > top_k:
                items = items[:top_k - 1] + [merged]  # top_k 里给合并项留一个位置
        return items[:top_k]

    def __len__(self):
        return len(self.vocab)

### `fit` 数数过程演示

假设迷你语料只有 2 句（已分词加标记）：

```
句1: <s> 我 爱 NLP </s>
句2: <s> 我 爱 语言 </s>
```

用 **trigram（n=3）** 训练，`fit` 内部发生的事：

**第 1 步：补足 `<s>`** —— 输入只有 1 个 `<s>`，n=3 的前缀需要 2 个词，先剥掉旧的再补齐：

```
句1 -> <s> <s> 我 爱 NLP </s>      （下标 0 1 2 3 4 5）
句2 -> <s> <s> 我 爱 语言 </s>
```

**第 3 步：滑动窗口数数** —— `i` 从 `n-1=2` 走到句子末尾，每个位置 i 上：
前缀 `ctx = seq[i-2:i]`（2 个词），当前词 `w = seq[i]`，两个计数器各 +1。
两句合并后的最终结果：

| 句 | i | 词 w_i | ctx（2 个词） | ngram_counts[ctx+w] | context_counts[ctx] |
|---|---|---|---|---|---|
| 1 | 2 | 我 | (\<s\>, \<s\>) | 1 | 1 |
| 1 | 3 | 爱 | (\<s\>, 我) | 1 | 1 |
| 1 | 4 | NLP | (我, 爱) | 1 | 1 |
| 1 | 5 | \</s\> | (爱, NLP) | 1 | 1 |
| 2 | 2 | 我 | (\<s\>, \<s\>) | **2** | **2** |
| 2 | 3 | 爱 | (\<s\>, 我) | **2** | **2** |
| 2 | 4 | 语言 | (我, 爱) | **1** | **2** |
| 2 | 5 | \</s\> | (爱, 语言) | 1 | 1 |

加粗的是两句重复出现、计数累加的位置。注意一个规律：
**`context_counts[ctx]` 永远等于「各种词接在这个 ctx 后面的次数总和」**，
比如 `context_counts[(我,爱)] = 2 = count(我,爱,NLP) + count(我,爱,语言)` ——
它就是公式的分母，`ngram_counts` 就是分子。

**第 4 步：加 `<unk>`** —— 数数结束后 `vocab.add("<unk>")`，此时词表为
`{<s>, 我, 爱, NLP, 语言, </s>, <unk>}`，即 **v = 7**。
`<unk>` 不出现在任何计数里（训练语料中没有生词），但推理时词表外的词
会被映射成它去查概率，所以分母的 v 必须给它留一个名额。

**数完后就能算概率了**。比如问 `p(语言 | 我, 爱)`（k=1，v=7）：

- 分子 = ngram_counts[(我,爱,语言)] + k = 1 + 1 = 2
- 分母 = context_counts[(我,爱)] + k×v = 2 + 1×7 = 9
- p = 2/9 ≈ 0.222

没见过的 `p(NLP | 我, 爱)` = (0+1) / (2+7) = 1/9 ≈ 0.111 —— 平滑保证它不是 0。
验证归一化：接 NLP、语言各 2/9，其余 5 个词（含 `<unk>`）各 1/9，
总和 = 2×(2/9) + 5×(1/9) = 1 ✓

In [15]:
K = 1  # 拉普拉斯平滑系数

bigram = NGramLM(n=2, k=K).fit(tokenized)
trigram = NGramLM(n=3, k=K).fit(tokenized)

print(f"词表大小 v = {len(bigram)}（bigram / trigram 相同）")
print(f"bigram  n-gram 种类数 = {len(bigram.ngram_counts)}")
print(f"trigram n-gram 种类数 = {len(trigram.ngram_counts)}")

# 手动验证几个条件概率
print(f"\np(爱 | 我)   = {bigram.prob('爱', '我'):.6f}")
print(f"p(中国 | 在)  = {bigram.prob('中国', '在'):.6f}")
print(f"p(经济 | 发展) = {bigram.prob('经济', '发展'):.6f}")

词表大小 v = 11875（bigram / trigram 相同）
bigram  n-gram 种类数 = 49939
trigram n-gram 种类数 = 66487

p(爱 | 我)   = 0.000084
p(中国 | 在)  = 0.001425
p(经济 | 发展) = 0.000082


## 5. 查候选：前一个词定了，下一个词有哪些选择？

In [16]:
for ctx in ["我", "中国", "发展"]:
    print(f"接在「{ctx}」后面的 Top 候选：")
    for w, p in bigram.candidates(ctx, top_k=8):
        print(f"   {w:<12} {p:.6f}")
    print()

接在「我」后面的 Top 候选：
   觉得           0.000420
   公司           0.000336
   代表           0.000252
   借            0.000168
   所查           0.000168
   的            0.000168
   这            0.000168
   <unk>(11864 词) 0.997645

接在「中国」后面的 Top 候选：
   真正           0.001313
   枸杞           0.001313
   最具           0.001313
   90%          0.001313
   的            0.001231
   纺织品          0.001066
   物流           0.001066
   <unk>(11756 词) 0.964397

接在「发展」后面的 Top 候选：
   的            0.004935
   空间           0.001892
   </s>         0.001727
   ，            0.001069
   战略           0.000987
   ,            0.000823
   中            0.000740
   <unk>(11796 词) 0.970225



## 6. 生成句子：贪心生成

每次从候选里挑概率最大的词接上去：
- 遇到 `</s>` 正常停止
- `max_length` 兜底，防止无限生成

另附 `sample=True` 的随机采样模式做对比（贪心往往一开口就是高频套话，采样更多样）。

In [17]:
def generate(lm: NGramLM, prompt: str, max_length: int = 20, sample: bool = False) -> str:
    """贪心（sample=False）或随机采样（sample=True）生成一句话。"""
    n = lm.n
    words = [w for w in jieba.cut(prompt) if w.strip()]
    ctx = ["<s>"] * (n - 1) + words
    result = list(words)

    for _ in range(max_length):
        context = tuple(ctx[-(n - 1):])
        cands = [(w, p) for w, p in lm.candidates(*context, top_k=None)
                 if not w.startswith("<unk>")]  # 去掉展示用的合并项
        if sample:
            total = sum(p for _, p in cands)
            r, acc = random.random() * total, 0.0
            w = cands[-1][0]
            for cand_w, p in cands:
                acc += p
                if acc >= r:
                    w = cand_w
                    break
        else:
            w = cands[0][0]
        if w == "</s>":
            break
        result.append(w)
        ctx.append(w)
    return "".join(result)


random.seed(42)
for prompt in ["今天", "中国", "经济"]:
    print(f"bigram  贪心  「{prompt}」 -> {generate(bigram, prompt)}")
    print(f"bigram  采样  「{prompt}」 -> {generate(bigram, prompt, sample=True)}")
    print(f"trigram 贪心  「{prompt}」 -> {generate(trigram, prompt)}")
    print(f"trigram 采样  「{prompt}」 -> {generate(trigram, prompt, sample=True)}")
    print()

bigram  贪心  「今天」 -> 今天各机构、本人所知情的范围内,论坛)（证券通”
bigram  采样  「今天」 -> 今天,论坛)、利益诱惑，也已经盯上的谨慎乐观
trigram 贪心  「今天」 -> 今天各机构的大举增持，期末已持有公司股票8965238股，绩优化工蓝筹正受到市场主力
trigram 采样  「今天」 -> 今天西藏天路(行情,论坛)承担了省、市行政体制改革的跟进，还要着力推进与

bigram  贪心  「中国」 -> 中国真正好喝的范围内,论坛)（证券通”
bigram  采样  「中国」 -> 中国电池产业分工与发力点,拥有，成为领涨龙头
trigram 贪心  「中国」 -> 中国的经济地位已远远超出学术辩论的范畴
trigram 采样  「中国」 -> 中国的广大农民来说，8元几成定局，摩根斯坦利甚至认为年底前人民币将升值至7.5

bigram  贪心  「经济」 -> 经济通”
bigram  采样  「经济」 -> 经济增长，将会获得重点是给公司业绩必将出现泡沬
trigram 贪心  「经济」 -> 经济的表现
trigram 采样  「经济」 -> 经济的高速发展



## 7. 评估

### 7.1 困惑度 PPL

$$\text{ppl} = 2^{-\frac{\log_2 p_1 + \log_2 p_2 + \cdots + \log_2 p_n}{n}}$$

n 指句子里参与评分的词数（含 `</s>`）；PPL 越低，模型觉得这句话越「像人话」。

### 7.2 整句概率

把每一对接法的条件概率相乘，看整句话像不像人话、连不连贯。

In [18]:
def sentence_prob(lm: NGramLM, sent: str) -> float:
    """整句概率 = 各条件概率连乘。"""
    words = [w for w in jieba.cut(sent) if w.strip()]
    seq = ["<s>"] * (lm.n - 1) + words + ["</s>"]
    p = 1.0
    for i in range(lm.n - 1, len(seq)):
        p *= lm.prob(seq[i], *seq[i - lm.n + 1: i])
    return p


def perplexity(lm: NGramLM, sent: str) -> float:
    """ppl = 2^(-(log2 p1 + ... + log2 pn) / n)，n 为参与评分的词数（含 </s>）。"""
    words = [w for w in jieba.cut(sent) if w.strip()]
    seq = ["<s>"] * (lm.n - 1) + words + ["</s>"]
    logps = [lm.log_prob(seq[i], *seq[i - lm.n + 1: i])
             for i in range(lm.n - 1, len(seq))]
    return 2 ** (-sum(logps) / len(logps))


good_sents = [
    "中国经济快速发展。",
    "今天天气很好，我们一起去公园玩。",
    "该公司发布了新的产品。",
]
bad_sents = [
    "数字经济香蕉跳跃在月亮的上面打篮球。",
    "火锅发动机温柔地讨论了数学作业。",
]

print(f"{'句子':<26} {'整句概率':>14} {'PPL(bigram)':>14} {'PPL(trigram)':>14}")
for s in good_sents + bad_sents:
    print(f"{s:<26} {sentence_prob(bigram, s):>14.3e} "
          f"{perplexity(bigram, s):>14.1f} {perplexity(trigram, s):>14.1f}")

句子                                   整句概率    PPL(bigram)   PPL(trigram)
中国经济快速发展。                       1.377e-22         4400.7         8360.5
今天天气很好，我们一起去公园玩。                5.136e-43         6990.1        12382.6
该公司发布了新的产品。                     2.723e-31         2489.4         5657.8
数字经济香蕉跳跃在月亮的上面打篮球。              5.414e-46        13035.8        12379.1
火锅发动机温柔地讨论了数学作业。                1.057e-41        12519.2        12430.7


In [19]:
# 留出最后 500 句做简易测试集：只用训练句重新训练，再在未见过的句子上算平均 PPL
train_sents, test_sents = tokenized[:-500], tokenized[-500:]
bigram_ho = NGramLM(n=2, k=K).fit(train_sents)
trigram_ho = NGramLM(n=3, k=K).fit(train_sents)
test_texts = ["".join(t[1:]).replace("</s>", "。") for t in test_sents]

def avg_ppl(lm: NGramLM, texts: list, k: int = 100) -> float:
    vals = [perplexity(lm, t) for t in random.sample(texts, min(k, len(texts)))]
    return sum(vals) / len(vals)

random.seed(0)
print(f"测试集平均 PPL  bigram  = {avg_ppl(bigram_ho, test_texts):.1f}")
print(f"测试集平均 PPL  trigram = {avg_ppl(trigram_ho, test_texts):.1f}")
print("\nPPL 越低越好；平滑系数 k、语料规模 MAX_FILES 都会影响它，可以调参对比。")

测试集平均 PPL  bigram  = 3853.4
测试集平均 PPL  trigram = 5424.9

PPL 越低越好；平滑系数 k、语料规模 MAX_FILES 都会影响它，可以调参对比。


## 小结

| 步骤 | 对应实现 |
| --- | --- |
| 切词 | jieba 分词 |
| 加标记 | `<s>` 开头，`</s>` 结尾（高阶模型训练时补足 n-1 个 `<s>`） |
| 数次数 | `NGramLM.fit`：`ngram_counts` / `context_counts` |
| 算概率 | `prob`：`(count + k) / (context_count + k × v)`，k=1 |
| 查候选 | `candidates`：按概率排序的下一个词 |
| 生成 | 贪心 / 采样，`max_length` 兜底 + `</s>` 停止 |
| 评估 | `perplexity`（PPL）+ `sentence_prob`（整句概率连乘） |

可以尝试的扩展：调大 `MAX_FILES`、调 `k`（如 0.5、1、5）看 PPL 变化；对比 n=2 与 n=3 的生成质量。